In [12]:
# Cellule 0 : créer corpus.txt à partir des explications (Markdown) du notebook du cours
import json
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
nb = json.loads((ROOT / "data" / "architecture_transformer.ipynb").read_text(encoding="utf-8"))

morceaux = []
for cellule in nb["cells"]:
    contenu = "".join(cellule["source"])
    if cellule["cell_type"] == "markdown" and "data:image" not in contenu:
        morceaux.append(contenu)

corpus = "\n\n".join(morceaux)
(ROOT / "data" / "corpus.txt").write_text(corpus, encoding="utf-8")
print("corpus.txt créé :", len(corpus), "caractères")

corpus.txt créé : 54716 caractères


In [13]:
from pathlib import Path
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(42)

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
texte = (ROOT / "data" / "corpus.txt").read_text(encoding="utf-8")

# Garder uniquement le roman (sans l'en-tête et la licence de Project Gutenberg)
debut = texte.find("*** START")
fin = texte.find("*** END")
if debut != -1 and fin != -1:
    texte = texte[texte.find("\n", debut) + 1:fin]
texte = texte[:300_000]   # largement assez pour un petit modèle

# Vocabulaire = l'ensemble des caractères du texte
caracteres = sorted(set(texte))
taille_vocabulaire = len(caracteres)
stoi = {c: i for i, c in enumerate(caracteres)}   # caractère -> identifiant
itos = {i: c for c, i in stoi.items()}            # identifiant -> caractère

def encoder(s):
    return [stoi[c] for c in s]

def decoder(ids):
    return "".join(itos[i] for i in ids)

tokens = torch.tensor(encoder(texte), dtype=torch.long)
coupure = int(0.9 * len(tokens))
train_tokens, val_tokens = tokens[:coupure], tokens[coupure:]

print("Caractères :", len(texte), "| taille du vocabulaire :", taille_vocabulaire)
print(texte[:300])

Caractères : 54716 | taille du vocabulaire : 120
# **Architecture Transformer**

### **Avant 2017 : RNN, LSTM, GRU**

## **1. RNN**

### Qu’est-ce qu’un RNN ?

Un **réseau de neurones récurrent** (*Recurrent Neural Network*) traite des données organisées en séquences : séries temporelles, textes, signaux, etc.

Il lit les éléments **dans l’ordre**


In [14]:
longueur = 64

def lot_aleatoire(donnees, taille_lot=32):
    """Tire des morceaux de 64 caractères au hasard. La cible = l'entrée décalée d'un caractère."""
    debuts = torch.randint(0, len(donnees) - longueur - 1, (taille_lot,))
    x = torch.stack([donnees[d:d + longueur] for d in debuts])
    y = torch.stack([donnees[d + 1:d + longueur + 1] for d in debuts])
    return x, y

x, y = lot_aleatoire(train_tokens, 1)
print("Entrée :", repr(decoder(x[0][:30].tolist())))
print("Cible  :", repr(decoder(y[0][:30].tolist())))

Entrée : 'odèle utilise le texte déjà di'
Cible  : 'dèle utilise le texte déjà dis'


In [15]:
class BlocGPT(nn.Module):
    def __init__(self, d_model=64, n_tetes=4):
        super().__init__()
        self.norm1 = nn.LayerNorm(d_model)
        self.attention = nn.MultiheadAttention(d_model, n_tetes, batch_first=True)
        self.norm2 = nn.LayerNorm(d_model)
        self.mlp = nn.Sequential(
            nn.Linear(d_model, 256),   # 64 -> 256
            nn.GELU(),
            nn.Linear(256, d_model),   # 256 -> 64
        )

    def forward(self, x):
        T = x.shape[1]
        # Masque causal : True = interdit de regarder cette position (le futur)
        masque = torch.triu(torch.ones(T, T, dtype=torch.bool, device=x.device), diagonal=1)

        h = self.norm1(x)                                        # 1. LayerNorm AVANT l'attention
        a, _ = self.attention(h, h, h, attn_mask=masque, need_weights=False)  # 2. attention causale
        x = x + a                                                # 3. connexion résiduelle
        x = x + self.mlp(self.norm2(x))                          # 4-6. LayerNorm, MLP, résiduelle
        return x

In [16]:
class MiniGPT(nn.Module):
    def __init__(self, taille_vocabulaire, longueur_max=64, d_model=64, n_tetes=4, n_blocs=2):
        super().__init__()
        self.tokens = nn.Embedding(taille_vocabulaire, d_model)     # caractère -> vecteur
        self.positions = nn.Embedding(longueur_max, d_model)        # position -> vecteur
        self.blocs = nn.Sequential(*[BlocGPT(d_model, n_tetes) for _ in range(n_blocs)])
        self.norm = nn.LayerNorm(d_model)
        self.tete = nn.Linear(d_model, taille_vocabulaire)          # vecteur -> score par caractère

    def forward(self, ids):
        T = ids.shape[1]
        h = self.tokens(ids) + self.positions(torch.arange(T, device=ids.device))
        h = self.blocs(h)
        return self.tete(self.norm(h))

modele = MiniGPT(taille_vocabulaire)

x, y = lot_aleatoire(train_tokens)
print("Identifiants en entrée :", x.shape)                     # (32, 64)
print("Logits                 :", modele(x).shape)             # (32, 64, taille_vocabulaire)
print("Cibles                 :", y.shape)                     # (32, 64)
print("Paramètres :", sum(p.numel() for p in modele.parameters()))

Identifiants en entrée : torch.Size([32, 64])
Logits                 : torch.Size([32, 64, 120])
Cibles                 : torch.Size([32, 64])
Paramètres : 119672


In [17]:
optimiseur = torch.optim.AdamW(modele.parameters(), lr=3e-3)

def perte_validation():
    modele.eval()
    with torch.no_grad():
        x, y = lot_aleatoire(val_tokens, 256)
        logits = modele(x)
        return F.cross_entropy(logits.reshape(-1, taille_vocabulaire), y.reshape(-1)).item()

for etape in range(1, 3001):
    modele.train()
    x, y = lot_aleatoire(train_tokens)
    logits = modele(x)                                        # (32, 64, vocab)
    perte = F.cross_entropy(logits.reshape(-1, taille_vocabulaire), y.reshape(-1))

    optimiseur.zero_grad()
    perte.backward()
    optimiseur.step()

    if etape % 500 == 0:
        print(f"Étape {etape} | train {perte.item():.3f} | val {perte_validation():.3f}")

Étape 500 | train 1.503 | val 1.992
Étape 1000 | train 1.006 | val 1.999
Étape 1500 | train 0.729 | val 2.201
Étape 2000 | train 0.671 | val 2.390
Étape 2500 | train 0.582 | val 2.644
Étape 3000 | train 0.501 | val 2.791


In [18]:
@torch.no_grad()
def generer(amorce, n=200, temperature=1.0):
    modele.eval()
    ids = torch.tensor([encoder(amorce)], dtype=torch.long)
    for _ in range(n):
        contexte = ids[:, -longueur:]                       # au plus les 64 derniers caractères
        logits = modele(contexte)[:, -1, :] / temperature   # scores du caractère suivant
        probas = torch.softmax(logits, dim=-1)
        suivant = torch.multinomial(probas, num_samples=1)  # tirage au sort
        ids = torch.cat([ids, suivant], dim=1)
    return decoder(ids[0].tolist())

print(generer("Le modèle ", 200))

Le modèle à jour un **emale** en «tchifichenche gradient : prédiction utilisé poutilisée pention plus entrées entrées dans la séquence. sura tertire des temporres.

---

### 5. Archin


Chaque exemple : `yU, co


In [19]:
for T in (0.5, 1.0, 1.5):
    torch.manual_seed(0)
    print(f"\n===== Température {T} =====")
    print(generer("Le modèle ", 200, temperature=T))


===== Température 0.5 =====
Le modèle prédiction |
| `attention_mask_tokens=True`** retire les temporelles, prétrochare et le confighe nécessaire d’entrée au texte :

- la couche linées dont la sa sau élet grainner les probables : inition

===== Température 1.0 =====
Le modèle possède ne sont :

| Paramètre | Signale, 16

Composantes construit un nouvellon plut.

Couche linélection compisentement le anci quel exclat ne selle gradient**.

----

## 3. Élémentisent PyTorch
hₜ 

===== Température 1.5 =====
Le modèle possède ne sonter la dernière.

---

## Étape 5 — Créer exemple, phrasenter le nombre.

Contrairement

Un **Lintint construction | Complementre d’entrée ;
- **P;
----

## 9, Si9}\der inm_le0 entrées ;


In [20]:
modele.eval()
x, _ = lot_aleatoire(val_tokens, 1)

x_modifie = x.clone()
x_modifie[0, -1] = (x[0, -1] + 1) % taille_vocabulaire   # on change le DERNIER caractère

with torch.no_grad():
    logits_1 = modele(x)
    logits_2 = modele(x_modifie)

print("Positions précédentes identiques :", torch.allclose(logits_1[:, :-1], logits_2[:, :-1], atol=1e-5))
print("Dernière position modifiée       :", not torch.allclose(logits_1[:, -1], logits_2[:, -1]))

Positions précédentes identiques : True
Dernière position modifiée       : True
